# Sea Ice Segmentation — Late Fusion (Long+Short Sparse-Aware BiLSTM) — CategoricalFocalLoss v4

**Input: CSV photon data only. No satellite imagery used.**

Two independent Sparse-Aware BiLSTM branches at different temporal scales.
**Key change from v2**: loss is `CategoricalFocalCrossentropy(alpha=[0.05, 0.50, 0.45], gamma=2.0)` — same alpha/gamma as `lstm_focal.ipynb`.

```
CSV window (B, K_LONG=48, F)
  |
  +---> LONG BRANCH  (K=48, Sparse-Aware BiLSTM, H=256, masked mean-pool)
  |        feat_long (B, 256)
  |             |
  |        cls_head_long --> logits_long (B, C) --> tile (B, C, 128, 128)
  |                                                        |
  |                                         alpha (learned)| (1-alpha)
  |                                                        |
  |                                       final logits <---+
  |                                                        |
  +---> SHORT BRANCH (K=16 central rows, Sparse-Aware BiLSTM, H=128)
           feat_short (B, 128)
                |
           cls_head_short --> logits_short (B, C) --> tile (B, C, 128, 128)
```

### Change from v2
| | v2 | **v4** |
|---|---|---|
| Loss | `FocalLoss(gamma=2.5, label_smoothing=0.05, class_weights)` | `CategoricalFocalLoss(alpha=[0.05,0.50,0.45], gamma=2.0)` |
| Architecture | 2 independent Sparse-Aware BiLSTM + decision blend | same |
| Two-phase training | Phase1 (long frozen) + Phase2 (full) | same |
| TTA | 6 passes (3 shifts × 2 noise) | same |

## 0. Setup

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

In [ ]:
import sys, subprocess
for pkg in ["tqdm"]:
    try: __import__(pkg)
    except ImportError: subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

In [ ]:
import json, math, random, re
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available(),
      "| device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

## 1. Config

In [ ]:
PROJECT_ROOT = Path.cwd()
EXP_NAME     = "latefusion_bilstm_v4"

MASK_DIR    = PROJECT_ROOT / "outputs_segmented"
CSV_DIR     = PROJECT_ROOT / "IS2_Corrected_data"
RUN_DIR     = PROJECT_ROOT / "runs" / EXP_NAME
PRIOR_LSTM  = PROJECT_ROOT / "runs" / "sparse_aware_lstm_ft_v1"
PRIOR_UNET  = PROJECT_ROOT / "runs" / "unet_imgonly_v1"
RUN_DIR.mkdir(parents=True, exist_ok=True)

SEED        = 42
NUM_CLASSES = 3
PATCH       = 128

K_LONG        = 48
HALF_LONG     = K_LONG // 2
HIDDEN_LONG   = 256
LAYERS_LONG   = 3
DROPOUT_LONG  = 0.3

K_SHORT       = 16
HALF_SHORT    = K_SHORT // 2
HIDDEN_SHORT  = 128
LAYERS_SHORT  = 2
DROPOUT_SHORT = 0.2

S_START = HALF_LONG - HALF_SHORT   # = 16
S_END   = HALF_LONG + HALF_SHORT   # = 32

PHASE1_EPOCHS = 10
PHASE2_EPOCHS = 20
EPOCHS        = PHASE1_EPOCHS + PHASE2_EPOCHS

LR           = 3e-4
LR_LONG      = LR / 10
WEIGHT_DECAY = 1e-3
PATIENCE     = 10
GRAD_CLIP    = 1.0
BATCH_SIZE   = 64
NUM_WORKERS  = 0
USE_AMP      = True

# ------- focal loss (per prof's notebook) -------
ALPHA = [0.05, 0.50, 0.45]   # [ice, thin_ice, water]
GAMMA = 2.0
# -------------------------------------------------

TTA_SHIFTS     = [-8, 0, 8]
TTA_NOISE_RUNS = 2
TTA_NOISE_STD  = 0.05

CLASS_NAMES  = ["ice", "thin_ice", "water"]
CLASS_COLORS = {0: (255, 0, 0), 1: (0, 0, 255), 2: (0, 255, 0)}

DROP_COLS = {
    "Unnamed: 0", "Ori_Id",
    "year", "month", "day", "hour", "minute", "second",
    "geometry", "pix_x", "pix_y", "label",
    "lat", "lon", "x", "y", "x_atc", "s_azi", "s_ele",
}

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

n_tta = len(TTA_SHIFTS) * TTA_NOISE_RUNS
print(f"Run dir     : {RUN_DIR}")
print(f"Long branch : K={K_LONG}, H={HIDDEN_LONG}, L={LAYERS_LONG}, sparse-aware")
print(f"Short branch: K={K_SHORT} (rows {S_START}:{S_END}), H={HIDDEN_SHORT}, L={LAYERS_SHORT}, sparse-aware")
print(f"Fusion      : pure late (decision-level alpha blend only)")
print(f"Focal loss  : alpha={ALPHA}, gamma={GAMMA}")
print(f"TTA passes  : {len(TTA_SHIFTS)} shifts x {TTA_NOISE_RUNS} noise = {n_tta}")

## 2. Manifest

In [ ]:
manifest_path_local = RUN_DIR / "manifest.csv"
manifest = None
for prior in [
    PRIOR_LSTM / "manifest.csv",
    PRIOR_UNET / "manifest.csv",
    PROJECT_ROOT / "runs" / "latefusion_bilstm_csvonly_v2" / "manifest.csv",
    PROJECT_ROOT / "runs" / "hybridfusion_csvonly_v2" / "manifest.csv",
]:
    if prior.exists():
        _m = pd.read_csv(prior)
        if "image_path" not in _m.columns:
            manifest = _m
            print(f"Loaded manifest from {prior}: {len(manifest):,} rows")
            break

if manifest is None:
    pat  = re.compile(r"^row(\d+)_(\d{8}T\d{6})_(T\d+[A-Z]+)_(gt[12]r)\.png$")
    rows = []
    for p in sorted(MASK_DIR.iterdir()):
        m = pat.match(p.name)
        if not m: continue
        rows.append({"filename": p.name, "row_idx": int(m.group(1)),
                     "date": m.group(2), "tile": m.group(3), "beam": m.group(4)})
    manifest = pd.DataFrame(rows)
    manifest.to_csv(manifest_path_local, index=False)
    print(f"Built manifest from mask dir: {len(manifest):,} rows")

manifest.head(3)

## 3. Link manifest to CSV files

In [ ]:
csv_files = sorted(CSV_DIR.glob("ATL03_*_done.csv"))
csv_meta  = []
for p in csv_files:
    parts = p.stem.split("_")
    csv_meta.append({"csv_path": str(p), "csv_name": p.name,
                     "tile": parts[3], "beam": parts[4]})
csv_meta = pd.DataFrame(csv_meta)
csv_meta["csv_id"] = csv_meta.index

manifest = manifest.merge(csv_meta[["tile", "beam", "csv_path", "csv_id"]],
                          on=["tile", "beam"], how="left")
assert manifest["csv_id"].notna().all()
manifest["csv_id"] = manifest["csv_id"].astype(int)
print(csv_meta[["csv_id", "tile", "beam", "csv_name"]])
print(f"manifest linked to {manifest['csv_id'].nunique()} unique CSVs")

## 4. Train / val / test split

In [ ]:
tiles_train = ["T02CNA", "T02CNC"]
tiles_test  = ["T03CWT"]

train_pool = manifest[manifest["tile"].isin(tiles_train)].reset_index(drop=True)
test_df    = manifest[manifest["tile"].isin(tiles_test) ].reset_index(drop=True)

rng          = np.random.RandomState(SEED)
val_idx      = rng.choice(len(train_pool), size=int(0.10 * len(train_pool)), replace=False)
val_mask_arr = np.zeros(len(train_pool), dtype=bool); val_mask_arr[val_idx] = True
train_df     = train_pool[~val_mask_arr].reset_index(drop=True)
val_df       = train_pool[ val_mask_arr].reset_index(drop=True)

print(f"Train: {len(train_df):,}   Val: {len(val_df):,}   Test: {len(test_df):,}")

## 5. CSV feature normalization

In [ ]:
raw_csvs = {}
for _, row in csv_meta.iterrows():
    raw_csvs[int(row["csv_id"])] = pd.read_csv(row["csv_path"])

first_id     = next(iter(raw_csvs))
feature_cols = [c for c in raw_csvs[first_id].columns if c not in DROP_COLS]
n_features   = len(feature_cols)
print(f"Using {n_features} features: {', '.join(feature_cols)}")

In [ ]:
prior_npy_dir = PRIOR_LSTM / "csv_normed"
prior_stats   = PRIOR_LSTM / "feature_stats.json"
local_npy_dir = RUN_DIR / "csv_normed"
local_npy_dir.mkdir(exist_ok=True)

prior_ok = (
    prior_stats.exists() and prior_npy_dir.exists()
    and all((prior_npy_dir / f"csv_{cid}.npy").exists() for cid in raw_csvs)
)

if prior_ok:
    print(f"Reusing cached arrays from {prior_npy_dir}")
    csv_features = {cid: np.load(prior_npy_dir / f"csv_{cid}.npy") for cid in raw_csvs}
    with open(prior_stats) as f: d = json.load(f)
    assert d["feature_cols"] == feature_cols
else:
    print("Computing normalization from scratch (training tiles only)")
    train_arrays = []
    for _, row in csv_meta.iterrows():
        if row["tile"] not in tiles_train: continue
        train_arrays.append(
            raw_csvs[int(row["csv_id"])][feature_cols].to_numpy(dtype=np.float32))
    train_concat = np.concatenate(train_arrays, axis=0)
    mu = np.nanmean(train_concat, axis=0).astype(np.float32)
    sd = np.nanstd(train_concat,  axis=0).astype(np.float32); sd[sd < 1e-6] = 1.0
    csv_features = {}
    for cid, df in raw_csvs.items():
        z = np.nan_to_num((df[feature_cols].to_numpy(dtype=np.float32) - mu) / sd)
        csv_features[cid] = z
        np.save(local_npy_dir / f"csv_{cid}.npy", z)
    with open(RUN_DIR / "feature_stats.json", "w") as f:
        json.dump({"feature_cols": feature_cols,
                   "mean": mu.tolist(), "std": sd.tolist()}, f, indent=2)

for cid, arr in csv_features.items():
    print(f"  csv_{cid}: {arr.shape}")

## 6. Mask helpers

In [ ]:
def mask_rgb_to_int(mask_rgb):
    out = np.full(mask_rgb.shape[:2], 255, dtype=np.uint8)
    out[(mask_rgb == [255, 0, 0]).all(axis=-1)] = 0
    out[(mask_rgb == [0, 0, 255]).all(axis=-1)] = 1
    out[(mask_rgb == [0, 255, 0]).all(axis=-1)] = 2
    return out

def int_mask_to_rgb(m):
    out = np.zeros((*m.shape, 3), dtype=np.uint8)
    for c, color in CLASS_COLORS.items(): out[m == c] = color
    return out

## 7. Class weights (reference only — CategoricalFocalLoss uses alpha instead)

In [ ]:
local_weights = RUN_DIR / "class_weights.json"
d = None
for pw in [PRIOR_UNET / "class_weights.json",
           PRIOR_LSTM / "class_weights.json", local_weights]:
    if pw.exists():
        with open(pw) as f: d = json.load(f)
        print(f"Loaded class weights from {pw}"); break

if d is None:
    n_sample = min(5000, len(train_df))
    counts   = np.zeros(NUM_CLASSES, dtype=np.int64)
    for fname in tqdm(train_df.sample(n_sample, random_state=SEED)["filename"],
                      desc="counting pixels"):
        m = mask_rgb_to_int(np.array(Image.open(MASK_DIR / fname).convert("RGB")))
        for c in range(NUM_CLASSES): counts[c] += int((m == c).sum())
    weights_arr = (counts.sum() / (NUM_CLASSES * counts.astype(np.float64))).astype(np.float32)
    d = {"counts": counts.tolist(), "weights": weights_arr.tolist(), "n_sample": int(n_sample)}
    with open(local_weights, "w") as f: json.dump(d, f, indent=2)

counts  = np.array(d["counts"],  dtype=np.int64)
weights = np.array(d["weights"], dtype=np.float32)
for c, name in enumerate(CLASS_NAMES):
    print(f"  {name:8s}  {counts[c]:>14,d} px ({100*counts[c]/counts.sum():5.2f}%)  weight={weights[c]:.3f}")
# CategoricalFocalLoss uses alpha=[0.05, 0.50, 0.45] instead of these weights.
print(f"\nUsing CategoricalFocalLoss: alpha={ALPHA}, gamma={GAMMA} (class weights above are for reference)")

## 8. Dataset — Sparse-Aware, CSV only

Returns long and short windows with their validity masks and density scalars.

In [ ]:
class LateFusionCSVDataset(Dataset):
    def __init__(self, df, csv_features, mask_dir, augment=False):
        self.df           = df.reset_index(drop=True)
        self.csv_features = csv_features
        self.mask_dir     = Path(mask_dir)
        self.augment      = augment

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        r      = self.df.iloc[i]
        feats  = self.csv_features[int(r["csv_id"])]
        n_rows, n_feat = feats.shape
        center = int(r["row_idx"])

        if self.augment:
            shift  = random.randint(-8, 8)
            center = max(HALF_LONG, min(n_rows - HALF_LONG - 1, center + shift))

        wl = np.zeros((K_LONG, n_feat), dtype=np.float32)
        vl = np.zeros((K_LONG,),        dtype=np.float32)
        for k in range(K_LONG):
            src = center - HALF_LONG + k
            if 0 <= src < n_rows: wl[k] = feats[src]; vl[k] = 1.0

        if self.augment and vl.sum() > 0:
            wl = wl + np.random.randn(*wl.shape).astype(np.float32) * 0.03 * vl[:, None]

        ws = wl[S_START:S_END]
        vs = vl[S_START:S_END]
        dl = float(vl.mean())
        ds = float(vs.mean())

        mask = mask_rgb_to_int(
            np.array(Image.open(self.mask_dir / r["filename"]).convert("RGB")))

        return (
            torch.from_numpy(wl), torch.from_numpy(vl),
            torch.tensor(dl, dtype=torch.float32),
            torch.from_numpy(ws), torch.from_numpy(vs),
            torch.tensor(ds, dtype=torch.float32),
            torch.from_numpy(mask).long(),
        )


_ds = LateFusionCSVDataset(train_df.head(8), csv_features, MASK_DIR)
_wl, _vl, _dl, _ws, _vs, _ds_val, _m = _ds[0]
print(f"win_long:  {tuple(_wl.shape)}  valid={int(_vl.sum())}/{K_LONG}  density={_dl.item():.2f}")
print(f"win_short: {tuple(_ws.shape)}  valid={int(_vs.sum())}/{K_SHORT}  density={_ds_val.item():.2f}")
print(f"mask:      {tuple(_m.shape)}  classes={torch.unique(_m).tolist()}")

## 9. DataLoaders

In [ ]:
train_ds = LateFusionCSVDataset(train_df, csv_features, MASK_DIR, augment=True)
val_ds   = LateFusionCSVDataset(val_df,   csv_features, MASK_DIR)
test_ds  = LateFusionCSVDataset(test_df,  csv_features, MASK_DIR)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=True,
                          persistent_workers=NUM_WORKERS > 0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True,
                          persistent_workers=NUM_WORKERS > 0)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True,
                          persistent_workers=NUM_WORKERS > 0)

print(f"steps/epoch  train: {len(train_loader)}  val: {len(val_loader)}  test: {len(test_loader)}")

## 10. Model — LateFusionCSVModel

Both branches are complete independent Sparse-Aware BiLSTM classifiers.
No feature sharing — only connection is the final alpha * logits_long + (1-alpha) * logits_short.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


class SparseAwareBiLSTMClassifier(nn.Module):
    """Complete Sparse-Aware BiLSTM with its own full classification head."""

    def __init__(self, n_features, window_k, hidden, layers, dropout, num_classes):
        super().__init__()
        self.pos_enc = nn.Embedding(window_k, n_features + 1)
        self.proj    = nn.Sequential(
            nn.Linear(n_features + 1, hidden), nn.LayerNorm(hidden), nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.lstm = nn.LSTM(
            input_size=hidden, hidden_size=hidden, num_layers=layers,
            batch_first=True, bidirectional=True,
            dropout=dropout if layers > 1 else 0.0,
        )
        self.merge = nn.Sequential(
            nn.Linear(hidden * 2 + 1, hidden), nn.LayerNorm(hidden),
            nn.ReLU(), nn.Dropout(dropout * 0.5),
        )
        self.cls_head = nn.Sequential(
            nn.Linear(hidden, hidden // 2),
            nn.LayerNorm(hidden // 2),
            nn.ReLU(),
            nn.Dropout(dropout * 0.3),
            nn.Linear(hidden // 2, num_classes),
        )

    def forward(self, x, valid, density):
        B, K, _ = x.shape
        x_aug   = torch.cat([x, valid.unsqueeze(-1)], dim=-1)
        pos     = self.pos_enc(torch.arange(K, device=x.device))
        x_aug   = x_aug + pos.unsqueeze(0) * valid.unsqueeze(-1)
        h       = self.proj(x_aug)
        out, _  = self.lstm(h)
        v_exp   = valid.unsqueeze(-1)
        pooled  = (out * v_exp).sum(1) / v_exp.sum(1).clamp(min=1.0)
        feat    = self.merge(torch.cat([pooled, density.unsqueeze(-1)], dim=-1))
        return self.cls_head(feat)


class LateFusionCSVModel(nn.Module):
    """Pure late fusion: two independent Sparse-Aware BiLSTM classifiers.
    No feature sharing. Final blend: alpha * logits_long + (1-alpha) * logits_short.
    """

    def __init__(self, n_features, num_classes=NUM_CLASSES, patch=PATCH,
                 hidden_long=HIDDEN_LONG, layers_long=LAYERS_LONG, dropout_long=DROPOUT_LONG,
                 hidden_short=HIDDEN_SHORT, layers_short=LAYERS_SHORT, dropout_short=DROPOUT_SHORT):
        super().__init__()
        self.patch = patch

        self.long_branch  = SparseAwareBiLSTMClassifier(
            n_features, K_LONG,  hidden_long,  layers_long,  dropout_long,  num_classes)
        self.short_branch = SparseAwareBiLSTMClassifier(
            n_features, K_SHORT, hidden_short, layers_short, dropout_short, num_classes)

        self.log_alpha = nn.Parameter(torch.zeros(1))

    def forward(self, wl, vl, dl, ws, vs, ds):
        logits_long  = self.long_branch(wl, vl, dl)
        logits_short = self.short_branch(ws, vs, ds)

        ll = logits_long[:, :, None, None].expand(-1, -1, self.patch, self.patch)
        ls = logits_short[:, :, None, None].expand(-1, -1, self.patch, self.patch)

        alpha = torch.sigmoid(self.log_alpha)
        return alpha * ll + (1.0 - alpha) * ls


model = LateFusionCSVModel(n_features=n_features).to(device)

with torch.no_grad():
    _wl = torch.zeros(2, K_LONG,  n_features, device=device)
    _vl = torch.ones(2, K_LONG,               device=device)
    _dl = torch.ones(2,                       device=device)
    _ws = torch.zeros(2, K_SHORT, n_features, device=device)
    _vs = torch.ones(2, K_SHORT,              device=device)
    _ds = torch.ones(2,                       device=device)
    _o  = model(_wl, _vl, _dl, _ws, _vs, _ds)
print(f"output shape : {tuple(_o.shape)}")

def n_params(m): return sum(p.numel() for p in m.parameters())
print(f"total params      : {n_params(model)/1e6:.2f} M")
print(f"  long branch     : {n_params(model.long_branch)/1e6:.2f} M")
print(f"  short branch    : {n_params(model.short_branch)/1e6:.2f} M")
print(f"  alpha init      : {torch.sigmoid(model.log_alpha).item():.4f}  (0.5 = equal weight)")

## 11. CategoricalFocalLoss (PyTorch port of Keras's `CategoricalFocalCrossentropy`)

Formula (matches Keras when `alpha` is given as a per-class vector):

    FL(p_t) = -alpha[true_class] * (1 - p_t)^gamma * log(p_t)

where `p_t` is the predicted probability for the true class.

In [ ]:
class CategoricalFocalLoss(nn.Module):
    def __init__(self, alpha, gamma=2.0, ignore_index=255):
        super().__init__()
        self.register_buffer("alpha", torch.tensor(alpha, dtype=torch.float32))
        self.gamma = float(gamma)
        self.ignore_index = ignore_index

    def forward(self, logits, target):
        valid       = target != self.ignore_index
        target_safe = target.clamp_min(0)

        log_probs = F.log_softmax(logits, dim=1)
        log_p_t   = log_probs.gather(1, target_safe.unsqueeze(1)).squeeze(1)
        p_t       = log_p_t.exp().clamp(min=1e-8, max=1.0 - 1e-8)

        alpha_t    = self.alpha.to(logits.device)[target_safe]
        focal_term = (1.0 - p_t).pow(self.gamma)
        per_pixel  = -alpha_t * focal_term * log_p_t

        per_pixel = per_pixel * valid.float()
        denom     = valid.float().sum().clamp_min(1.0)
        return per_pixel.sum() / denom


criterion = CategoricalFocalLoss(alpha=ALPHA, gamma=GAMMA).to(device)
with torch.no_grad():
    _l = torch.randn(2, NUM_CLASSES, PATCH, PATCH, device=device)
    _y = torch.zeros(2, PATCH, PATCH, dtype=torch.long, device=device)
    print(f"CategoricalFocalLoss: alpha={ALPHA}, gamma={GAMMA}")
    print(f"  sanity loss={criterion(_l, _y).item():.4f}")

## 12. Training utilities — IoU, evaluate, TTA

TTA: **6 passes (3 shifts x 2 noise runs)**. Shift applied to the full K_LONG window;
short window re-extracted from the shifted long window.

In [ ]:
class IoUAccumulator:
    def __init__(self, num_classes=3):
        self.num_classes = num_classes; self.reset()

    def reset(self):
        self.inter = np.zeros(self.num_classes, dtype=np.int64)
        self.union = np.zeros(self.num_classes, dtype=np.int64)
        self.cm    = np.zeros((self.num_classes, self.num_classes), dtype=np.int64)

    def update(self, preds, targets):
        p = preds.detach().cpu().numpy().ravel()
        t = targets.detach().cpu().numpy().ravel()
        for c in range(self.num_classes):
            self.inter[c] += int(np.logical_and(p == c, t == c).sum())
            self.union[c] += int(np.logical_or (p == c, t == c).sum())
        self.cm += np.bincount(self.num_classes * t + p,
                               minlength=self.num_classes**2
                               ).reshape(self.num_classes, self.num_classes)

    def per_class_iou(self): return self.inter / np.maximum(self.union, 1)
    def miou(self):          return float(self.per_class_iou().mean())
    def pixel_accuracy(self):
        return float(np.diag(self.cm).sum() / max(self.cm.sum(), 1))


def _unpack(batch, device):
    wl, vl, dl, ws, vs, ds, y = batch
    return (t.to(device, non_blocking=True) for t in (wl, vl, dl, ws, vs, ds, y))


def evaluate(model, loader, criterion, device):
    model.eval()
    acc = IoUAccumulator(NUM_CLASSES)
    loss_sum, n = 0.0, 0
    with torch.no_grad():
        for batch in tqdm(loader, desc="eval", leave=False):
            wl, vl, dl, ws, vs, ds, y = _unpack(batch, device)
            with torch.amp.autocast("cuda", enabled=USE_AMP):
                logits = model(wl, vl, dl, ws, vs, ds)
                loss   = criterion(logits, y)
            acc.update(logits.argmax(1), y)
            loss_sum += loss.item() * wl.size(0); n += wl.size(0)
    return {"loss": loss_sum / max(n, 1), "miou": acc.miou(),
            "per_iou": acc.per_class_iou().tolist(),
            "pix_acc": acc.pixel_accuracy(), "cm": acc.cm.tolist()}


def _shift_long(wl, vl, shift):
    if shift == 0: return wl, vl
    wls = torch.roll(wl, shift, dims=1)
    vls = torch.roll(vl, shift, dims=1)
    if shift > 0: wls[:, :shift] = 0.0; vls[:, :shift] = 0.0
    else:         wls[:, shift:] = 0.0; vls[:, shift:] = 0.0
    return wls, vls


def tta_predict_batch(model, wl, vl, device,
                      shifts=TTA_SHIFTS, noise_std=TTA_NOISE_STD,
                      noise_runs=TTA_NOISE_RUNS):
    all_probs = []
    model.eval()
    with torch.no_grad():
        for shift in shifts:
            wl_s, vl_s = _shift_long(wl, vl, shift)
            ws_s = wl_s[:, S_START:S_END]; vs_s = vl_s[:, S_START:S_END]
            dl_s = vl_s.mean(dim=1); ds_s = vs_s.mean(dim=1)
            for run in range(noise_runs):
                wla = wl_s.clone()
                if run > 0 and noise_std > 0:
                    wla = wla + torch.randn_like(wla) * noise_std * vl_s.unsqueeze(-1)
                wsa = wla[:, S_START:S_END]
                with torch.amp.autocast("cuda", enabled=USE_AMP):
                    probs = F.softmax(model(wla, vl_s, dl_s, wsa, vs_s, ds_s), dim=1)
                all_probs.append(probs)
    return torch.stack(all_probs).mean(0)


def evaluate_tta(model, loader, criterion, device):
    n_passes = len(TTA_SHIFTS) * TTA_NOISE_RUNS
    model.eval()
    acc = IoUAccumulator(NUM_CLASSES)
    loss_sum, n = 0.0, 0
    with torch.no_grad():
        for batch in tqdm(loader, desc=f"TTA ({n_passes} passes)", leave=False):
            wl, vl, dl, ws, vs, ds, y = _unpack(batch, device)
            avg_probs = tta_predict_batch(model, wl, vl, device)
            acc.update(avg_probs.argmax(1), y)
            with torch.amp.autocast("cuda", enabled=USE_AMP):
                loss = criterion(model(wl, vl, dl, ws, vs, ds), y)
            loss_sum += loss.item() * wl.size(0); n += wl.size(0)
    return {"loss": loss_sum / max(n, 1), "miou": acc.miou(),
            "per_iou": acc.per_class_iou().tolist(),
            "pix_acc": acc.pixel_accuracy(), "cm": acc.cm.tolist()}


print(f"TTA: {len(TTA_SHIFTS)} shifts x {TTA_NOISE_RUNS} noise = {len(TTA_SHIFTS)*TTA_NOISE_RUNS} passes")

## 13. Two-phase fine-tuning setup

**Phase 1**: freeze long branch. Train only short branch + alpha.
**Phase 2**: unfreeze all; long branch at `LR/10`.

In [ ]:
def freeze_long_branch(model):
    for p in model.long_branch.parameters(): p.requires_grad = False

def unfreeze_all(model):
    for p in model.parameters(): p.requires_grad = True

def make_optimizer_phase1(model):
    return torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad], lr=LR, weight_decay=WEIGHT_DECAY)

def make_optimizer_phase2(model):
    long_ids = {id(p) for p in model.long_branch.parameters()}
    long_p   = [p for p in model.parameters() if id(p) in long_ids]
    rest_p   = [p for p in model.parameters() if id(p) not in long_ids]
    return torch.optim.AdamW(
        [{"params": long_p, "lr": LR_LONG}, {"params": rest_p, "lr": LR}],
        weight_decay=WEIGHT_DECAY)


freeze_long_branch(model)
frozen    = sum(p.numel() for p in model.parameters() if not p.requires_grad)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Phase 1  frozen (long branch): {frozen/1e6:.2f}M  trainable: {trainable/1e6:.2f}M")

## 14. Training loop

In [ ]:
scaler    = torch.amp.GradScaler("cuda", enabled=USE_AMP)
optimizer = make_optimizer_phase1(model)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=LR,
    steps_per_epoch=len(train_loader), epochs=PHASE1_EPOCHS,
    pct_start=0.3, div_factor=25.0, final_div_factor=1e4, anneal_strategy="cos",
)

best_miou     = -1.0
patience_left = PATIENCE
log_path      = RUN_DIR / "metrics.csv"
ckpt_path     = RUN_DIR / "best.pt"
log           = []
phase         = 1

for epoch in range(1, EPOCHS + 1):

    if epoch == PHASE1_EPOCHS + 1 and phase == 1:
        print("\n" + "=" * 60)
        print("PHASE 2 — unfreezing long branch, differential LR")
        print("=" * 60)
        unfreeze_all(model)
        optimizer     = make_optimizer_phase2(model)
        scheduler     = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=PHASE2_EPOCHS)
        patience_left = PATIENCE; phase = 2
        print(f"long LR={LR_LONG:.1e}  short+alpha LR={LR:.1e}\n")

    model.train()
    tls, n = 0.0, 0
    pbar = tqdm(train_loader, desc=f"epoch {epoch:02d}/{EPOCHS} [phase {phase}]", leave=False)
    for batch in pbar:
        wl, vl, dl, ws, vs, ds, y = _unpack(batch, device)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", enabled=USE_AMP):
            logits = model(wl, vl, dl, ws, vs, ds)
            loss   = criterion(logits, y)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer); scaler.update()
        if phase == 1: scheduler.step()
        tls += loss.item() * wl.size(0); n += wl.size(0)
        pbar.set_postfix(loss=f"{loss.item():.3f}", lr=f"{optimizer.param_groups[-1]['lr']:.2e}")

    if phase == 2: scheduler.step()

    train_loss = tls / max(n, 1)
    val_m      = evaluate(model, val_loader, criterion, device)
    alpha_val  = torch.sigmoid(model.log_alpha).item()

    print(f"epoch {epoch:02d}  train={train_loss:.4f}  val_loss={val_m['loss']:.4f}  "
          f"val_mIoU={val_m['miou']:.4f}  per={[f'{v:.3f}' for v in val_m['per_iou']]}  "
          f"pix_acc={val_m['pix_acc']:.4f}  alpha={alpha_val:.3f}  "
          f"lr={optimizer.param_groups[-1]['lr']:.2e}")

    log.append({"epoch": epoch, "phase": phase, "train_loss": train_loss,
                "val_loss": val_m["loss"], "val_miou": val_m["miou"],
                "iou_ice": val_m["per_iou"][0], "iou_thin": val_m["per_iou"][1],
                "iou_water": val_m["per_iou"][2], "pix_acc": val_m["pix_acc"],
                "alpha": alpha_val, "lr": optimizer.param_groups[-1]["lr"]})
    pd.DataFrame(log).to_csv(log_path, index=False)

    if val_m["miou"] > best_miou:
        best_miou = val_m["miou"]
        torch.save({"epoch": epoch, "phase": phase, "model_state": model.state_dict(),
                    "val_metrics": val_m, "n_features": n_features, "alpha": alpha_val,
                    "focal": {"alpha": ALPHA, "gamma": GAMMA},
                    "tta_config": {"shifts": TTA_SHIFTS, "noise_std": TTA_NOISE_STD,
                                   "noise_runs": TTA_NOISE_RUNS}}, ckpt_path)
        patience_left = PATIENCE
        print(f"  -> saved best ({best_miou:.4f}) alpha={alpha_val:.3f}")
    else:
        patience_left -= 1
        if patience_left <= 0: print(f"  -> early stop"); break

print(f"\nBest val mIoU: {best_miou:.4f}")

## 15. Test evaluation — standard and TTA

In [ ]:
ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
model.load_state_dict(ckpt["model_state"])
print(f"Loaded checkpoint  epoch {ckpt['epoch']} (phase {ckpt['phase']})  "
      f"val mIoU={ckpt['val_metrics']['miou']:.4f}  alpha={ckpt['alpha']:.3f}")

print("\n--- Standard ---")
test_std = evaluate(model, test_loader, criterion, device)
print(f"mIoU={test_std['miou']:.4f}   pix_acc={test_std['pix_acc']:.4f}")
for cn, iou in zip(CLASS_NAMES, test_std["per_iou"]): print(f"  {cn:8s}  {iou:.4f}")
with open(RUN_DIR / "test_metrics_std.json", "w") as f:
    json.dump({k: v for k, v in test_std.items() if k != "cm"}, f, indent=2)

n_tta = len(TTA_SHIFTS) * TTA_NOISE_RUNS
print(f"\n--- TTA ({len(TTA_SHIFTS)} shifts x {TTA_NOISE_RUNS} noise = {n_tta} passes) ---")
test_tta = evaluate_tta(model, test_loader, criterion, device)
print(f"mIoU={test_tta['miou']:.4f}   pix_acc={test_tta['pix_acc']:.4f}")
for cn, iou in zip(CLASS_NAMES, test_tta["per_iou"]): print(f"  {cn:8s}  {iou:.4f}")
with open(RUN_DIR / "test_metrics_tta.json", "w") as f:
    json.dump({k: v for k, v in test_tta.items() if k != "cm"}, f, indent=2)

print("\n" + "=" * 55)
print(f"{'Metric':<20} {'Standard':>10} {'TTA':>10} {'Delta':>10}")
print("-" * 55)
for lbl, ks in [("mIoU", "miou"), ("Pixel accuracy", "pix_acc")]:
    s, t = test_std[ks], test_tta[ks]
    print(f"  {lbl:<18} {s:>10.4f} {t:>10.4f} {t-s:>+10.4f}")
print("-" * 55)
for cn, is_, it in zip(CLASS_NAMES, test_std["per_iou"], test_tta["per_iou"]):
    print(f"  IoU {cn:<14} {is_:>10.4f} {it:>10.4f} {it-is_:>+10.4f}")
print("=" * 55)
print(f"\nalpha at test: {torch.sigmoid(model.log_alpha).item():.4f}")
print("  alpha -> 1.0: long-range (K=48) branch dominates")
print("  alpha -> 0.0: short-range (K=16) branch dominates")
print("  alpha ~  0.5: both temporal scales contribute equally")

## 16. Confusion Matrices

In [ ]:
DISPLAY_NAMES = ["thick ice", "thin ice", "water"]

def plot_cm_pct(cm_list, title, ax):
    cm      = np.array(cm_list, dtype=np.float64)
    row_sum = cm.sum(axis=1, keepdims=True)
    cm_pct  = np.where(row_sum > 0, cm / row_sum * 100, 0.0)
    im = ax.imshow(cm_pct, vmin=0, vmax=100, cmap="Blues")
    ax.set_title(title, fontsize=11, pad=8)
    ax.set_ylabel("Actual", fontsize=9); ax.set_xlabel("Predicted", fontsize=9)
    ax.set_xticks(range(3)); ax.set_yticks(range(3))
    ax.set_xticklabels([f"Pred {n}" for n in DISPLAY_NAMES], fontsize=7)
    ax.set_yticklabels(DISPLAY_NAMES, fontsize=8)
    for i in range(3):
        for j in range(3):
            v = cm_pct[i, j]
            ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=10,
                    fontweight="bold", color="white" if v > 60 else "black")
    return im

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
fig.subplots_adjust(wspace=0.38)
im0 = plot_cm_pct(test_std["cm"], "Standard inference", axes[0])
im1 = plot_cm_pct(test_tta["cm"], f"TTA ({n_tta} passes)", axes[1])
plt.colorbar(im0, ax=axes[0], fraction=0.046)
plt.colorbar(im1, ax=axes[1], fraction=0.046)
plt.suptitle(f"Late Fusion v4 (Sparse-Aware BiLSTM, CategoricalFocalLoss alpha={ALPHA}) — Test T03CWT",
             fontsize=10, y=1.02)
plt.tight_layout()
plt.savefig(RUN_DIR / "confmat_pct.png", dpi=150, bbox_inches="tight")
plt.show()

## 17. Sample predictions

In [ ]:
model.eval()
N_SAMPLES  = 30
N_COLS     = 5
N_ROWS_VIS = math.ceil(N_SAMPLES / N_COLS)
sample_vis = test_df.sample(n=N_SAMPLES, random_state=SEED).reset_index(drop=True)

fig, axes = plt.subplots(N_ROWS_VIS, N_COLS * 3,
                         figsize=(N_COLS * 4.5, N_ROWS_VIS * 1.7))
fig.subplots_adjust(wspace=0.03, hspace=0.22)

with torch.no_grad():
    for idx in range(N_SAMPLES):
        rg, cg = divmod(idx, N_COLS)
        ax_gt, ax_std, ax_tta = (axes[rg, cg*3 + k] for k in range(3))

        r      = sample_vis.iloc[idx]
        feats  = csv_features[int(r["csv_id"])]
        n_rows = feats.shape[0]; center = int(r["row_idx"])

        wl_np = np.zeros((K_LONG, n_features), dtype=np.float32)
        vl_np = np.zeros((K_LONG,),            dtype=np.float32)
        for k in range(K_LONG):
            src = center - HALF_LONG + k
            if 0 <= src < n_rows: wl_np[k] = feats[src]; vl_np[k] = 1.0
        ws_np = wl_np[S_START:S_END]; vs_np = vl_np[S_START:S_END]

        gt = mask_rgb_to_int(
            np.array(Image.open(MASK_DIR / r["filename"]).convert("RGB")))

        def _to(x): return torch.from_numpy(x)[None].to(device)
        wl_t = _to(wl_np); vl_t = _to(vl_np)
        ws_t = _to(ws_np); vs_t = _to(vs_np)
        dl_t = vl_t.mean(dim=1); ds_t = vs_t.mean(dim=1)

        with torch.amp.autocast("cuda", enabled=USE_AMP):
            pred_std = model(wl_t, vl_t, dl_t, ws_t, vs_t, ds_t).argmax(1)[0].cpu().numpy()
        pred_tta = tta_predict_batch(model, wl_t, vl_t, device).argmax(1)[0].cpu().numpy()

        for ax, im in [(ax_gt, int_mask_to_rgb(gt)),
                       (ax_std, int_mask_to_rgb(pred_std)),
                       (ax_tta, int_mask_to_rgb(pred_tta))]:
            ax.imshow(im); ax.axis("off")
        ax_gt.set_title(f"#{idx+1}", fontsize=5, pad=1)
        if idx == 0:
            for ax, lbl in zip([ax_gt, ax_std, ax_tta], ["GT", "Std", "TTA"]):
                ax.set_xlabel(lbl, fontsize=6, labelpad=2)

patches = [mpatches.Patch(facecolor=np.array(c)/255, label=nm)
           for nm, c in zip(CLASS_NAMES, CLASS_COLORS.values())]
fig.legend(handles=patches, loc="lower center", ncol=3, fontsize=8, bbox_to_anchor=(0.5, -0.01))
fig.suptitle(
    f"Late Fusion v4 (CSV only) | T03CWT | mIoU={best_miou:.4f} | "
    f"alpha={torch.sigmoid(model.log_alpha).item():.3f}",
    fontsize=9, y=1.005)
plt.tight_layout()
plt.savefig(RUN_DIR / "sample_predictions.png", dpi=110, bbox_inches="tight")
plt.show()

## 18. Learning Curves

In [ ]:
log_df = pd.read_csv(log_path)
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))

ax = axes[0]
ax.plot(log_df["epoch"], log_df["train_loss"], label="Train", linewidth=2)
ax.plot(log_df["epoch"], log_df["val_loss"],   label="Val",   linewidth=2)
ax.axvline(PHASE1_EPOCHS+0.5, color="gray", linestyle="--", linewidth=1, label=f"Phase 2")
ax.set_xlabel("Epoch"); ax.set_title("Loss"); ax.legend(fontsize=8); ax.grid(alpha=0.3)

ax = axes[1]
ax.plot(log_df["epoch"], log_df["val_miou"],  label="mIoU",    linewidth=2, color="tab:green")
ax.plot(log_df["epoch"], log_df["iou_ice"],   label="ice",     linestyle="--", linewidth=1.5)
ax.plot(log_df["epoch"], log_df["iou_thin"],  label="thin",    linestyle="--", linewidth=1.5)
ax.plot(log_df["epoch"], log_df["iou_water"], label="water",   linestyle="--", linewidth=1.5)
ax.axvline(PHASE1_EPOCHS+0.5, color="gray", linestyle="--", linewidth=1)
ax.set_xlabel("Epoch"); ax.set_title("Val mIoU"); ax.legend(fontsize=8); ax.grid(alpha=0.3)

ax = axes[2]
ax.plot(log_df["epoch"], log_df["alpha"], linewidth=2, color="tab:purple")
ax.axhline(0.5, color="gray", linestyle=":", linewidth=1)
ax.axvline(PHASE1_EPOCHS+0.5, color="gray", linestyle="--", linewidth=1)
ax.set_xlabel("Epoch")
ax.set_title("Blend alpha\n(1=long K=48, 0=short K=16)")
ax.set_ylim(0, 1); ax.grid(alpha=0.3)

plt.suptitle(f"Late Fusion v4 (Sparse-Aware BiLSTM, CategoricalFocalLoss alpha={ALPHA}) — Learning Curves",
             fontsize=11)
plt.tight_layout()
plt.savefig(RUN_DIR / "learning_curves.png", dpi=150)
plt.show()

## 19. Classification Report

In [ ]:
from sklearn.metrics import classification_report

def report_from_cm(cm_list, names=DISPLAY_NAMES):
    cm = np.array(cm_list, dtype=np.int64)
    y_true, y_pred = [], []
    for ti in range(len(names)):
        for pi in range(len(names)):
            c = int(cm[ti, pi]); y_true.extend([ti]*c); y_pred.extend([pi]*c)
    return classification_report(y_true, y_pred, target_names=names, digits=4, zero_division=0)

print("=" * 55)
print("Classification Report — Standard")
print("=" * 55)
print(report_from_cm(test_std["cm"]))
print("=" * 55)
print(f"Classification Report — TTA ({n_tta} passes)")
print("=" * 55)
print(report_from_cm(test_tta["cm"]))

## Done — Artifacts in `runs/latefusion_bilstm_v4/`

* `best.pt`               — best-by-val checkpoint
* `metrics.csv`           — per-epoch training curves (loss, mIoU, alpha, lr)
* `test_metrics_std.json` — standard test mIoU / per-class IoU / pix_acc
* `test_metrics_tta.json` — TTA test metrics (6 passes)
* `confmat_pct.png`       — side-by-side confusion matrices (std vs TTA)
* `sample_predictions.png`— 30 test patches: GT | std pred | TTA pred
* `learning_curves.png`   — loss, mIoU, alpha over epochs

### Key difference vs v2
Loss changed from `FocalLoss(gamma=2.5, label_smoothing=0.05, class_weights)` to
`CategoricalFocalLoss(alpha=[0.05, 0.50, 0.45], gamma=2.0)` — the per-class alpha
matches the professor's Keras notebook and the `lstm_focal.ipynb` baseline.

### Interpreting alpha
Because the two branches are fully independent, alpha directly quantifies
whether long-range temporal context (K=48) or local context (K=16) is more
informative for sea ice classification on this dataset.